# TP vs FP: which features tell them apart?

A template analysis with tessa: raw per-asset parquet data plus an Excel label
sheet, taken to a verdict ("are the classes separable at all?"), a feature
ranking, and a shareable HTML report.

**How to use it.** As shipped, it runs end to end on generated example data
(`USE_EXAMPLE_DATA = True`). To run it on your data, fill in every `TODO` in
the settings cell and set `USE_EXAMPLE_DATA = False`. The rest of the notebook
reads only those settings.

1. Settings
2. The data store: which assets, sources and periods exist
3. The label sheet
4. Matching labels to data: drop events the store cannot serve
5. A look at raw events
6. Features: one row per event
7. Analyses
8. Save, report, dashboard

## 1. Settings

In [ ]:
from pathlib import Path

USE_EXAMPLE_DATA = True  # TODO: set to False once the paths below point at your data

# Notebooks run with the notebook's folder as working directory.
PROJECT_ROOT = Path.cwd().parent

# --- Data store ---------------------------------------------------------------
DATA_ROOT = Path("/path/to/data")  # TODO: folder with one sub-folder per asset
TIMESTAMP_COL = "timestamp"  # TODO: name of the time column inside the parquet files

# --- Label sheet --------------------------------------------------------------
LABELS_XLSX = Path("/path/to/labels.xlsx")  # TODO
LABELS_SHEET = 0  # TODO: sheet name, or 0-based sheet index
COLUMN_MAP = {  # TODO: your Excel headers -> the names tessa expects
    "Asset": "asset_id",
    "Start time": "start",
    "End time": "end",
    "Label": "class",
    "Replacement type": "replacement_type",  # optional extra column, passed through
}

# --- The question -------------------------------------------------------------
CLASSES = ["TP", "FP"]  # TODO: the classes to compare
STRATIFY_BY = "replacement_type"  # TODO: a label column to split the analysis by, or None
SIGNALS = None  # TODO: e.g. ["temperature", "vibration"]; None = every column in the files
PLOT_SIGNAL = None  # TODO: the signal to eyeball in section 5; None = the first one
MIN_COVERAGE = 0.9  # drop events whose window the data files cover by less than 90%

OUTPUT_DIR = PROJECT_ROOT / "outputs"
RANDOM_STATE = 7

if USE_EXAMPLE_DATA:
    from tessa.examples import make_example_dataset

    example = make_example_dataset(PROJECT_ROOT / "example_data")
    DATA_ROOT, LABELS_XLSX, COLUMN_MAP = example.data_root, example.labels_path, example.column_map
    LABELS_SHEET = "labels"
    PLOT_SIGNAL = "temperature"

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import polars as pl

import tessa
from tessa import Config, Dataset, Run
from tessa.analysis import NullPolicy
from tessa.dataset import file_catalog
from tessa.features import FeatureRegistry, to_period
from tessa.features.builtins import make_first_difference, make_rolling_std
from tessa.labels import ExcelLabelSource

pl.Config.set_tbl_rows(20)
print("tessa", tessa.__version__)

## 2. The data store

`Config` describes the on-disk layout. The defaults expect
`<DATA_ROOT>/<asset_id>/<source>_<start>_<end>.parquet`, with dates as `YYMMDD`
or `YYYYMMDD`. Files sharing a source name are concatenated over time; different
sources are outer-joined on the timestamp. If your files are laid out
differently, set `asset_subdir` or `filename_pattern` (a regex with named groups
`source`, `start`, `end`).

The catalogue below is built from file *names* only, so it is instant even on a
large store: it shows what exists before anything is read.

In [ ]:
cfg = Config(
    data_root=DATA_ROOT,
    timestamp_col=TIMESTAMP_COL,
    output_dir=OUTPUT_DIR,
    random_state=RANDOM_STATE,
    # asset_subdir="input",  # files in <asset>/input/ instead of <asset>/
    # filename_pattern=r"^(?P<source>.+)_(?P<start>\d{8})_(?P<end>\d{8})\.parquet$",
)
ds = Dataset(cfg)

catalog = file_catalog(cfg)
print(f"{len(ds.assets)} asset folders, {catalog.height} parquet files under {DATA_ROOT}")
catalog.group_by("asset_id", "source").agg(
    n_files=pl.len(),
    first_day=pl.col("start").min(),
    last_day=pl.col("end").max(),
).sort("asset_id", "source")

In [ ]:
# The columns of one asset, across all its sources (schemas only, no data read)
asset = catalog["asset_id"][0]
raw_columns = ds.channels(asset)
signals = SIGNALS or [c for c in raw_columns if c != cfg.timestamp_col]
print(f"{asset}: {raw_columns}")
print("signals analysed:", signals)

## 3. The label sheet

`ExcelLabelSource` reads the sheet, renames columns via `COLUMN_MAP` and parses
`start` / `end` as datetimes. Every other column is kept as event metadata.

In [ ]:
labels = ExcelLabelSource(LABELS_XLSX, sheet=LABELS_SHEET, column_map=COLUMN_MAP).load(cfg)

# Extra label columns travel with every event as label metadata, whatever their
# dtype: a numeric one (a severity score, an ID) is carried into the feature table
# as-is and is never aggregated or used as a feature (see sections 6 and 7).
extras = [c for c in labels.columns if c not in (cfg.asset_col, "start", "end", cfg.class_col)]

print(f"{labels.height} label rows, extra columns: {extras}")
labels.group_by(cfg.class_col).len().sort(cfg.class_col)

## 4. Matching labels to data

Label sheets and data exports are maintained separately, so some rows usually
point at assets that were never exported or periods outside the export. Building
events for them would fail, so first ask the store what it can serve.

`check="rows"` also reads the timestamp column of the matching files and confirms
the window holds data. `check="files"` decides from file names alone: faster on
a big store, but it misses gaps inside a file.

In [ ]:
availability = ds.availability(labels, check="rows")
availability.group_by("data_reason").len().sort("len", descending=True)

In [ ]:
# The rows that will be dropped, and why
availability.filter(
    ~pl.col("data_available") | (pl.col("data_coverage") < MIN_COVERAGE)
).select(cfg.asset_col, "start", "end", cfg.class_col, "data_coverage", "data_reason")

In [ ]:
# Keep what the store can serve, then restrict to the classes being compared.
# The warning lists what was dropped, so the label table never shrinks silently.
labels_ok = ds.available(labels, check="rows", min_coverage=MIN_COVERAGE)
labels_ok = labels_ok.filter(pl.col(cfg.class_col).is_in(CLASSES))

print(f"{labels_ok.height} of {labels.height} label rows kept")
by = [cfg.class_col, STRATIFY_BY] if STRATIFY_BY else [cfg.class_col]
labels_ok.group_by(by).len().sort(by)

## 5. A look at raw events

`ds.events()` returns one lazy frame per event, with the label columns attached.
Nothing is read until a frame is collected. Plot a few raw events per class
before trusting any statistic.

In [ ]:
events = ds.events(labels_ok)  # {event_id: LazyFrame}, in label-row order
event_index = labels_ok.with_columns(event_id=pl.Series(list(events)))

plot_signal = PLOT_SIGNAL or signals[0]
N_PER_CLASS = 5

fig, axes = plt.subplots(1, len(CLASSES), figsize=(5 * len(CLASSES), 3.5), sharey=True)
for ax, cls in zip(np.atleast_1d(axes), CLASSES):
    ids = event_index.filter(pl.col(cfg.class_col) == cls)["event_id"].head(N_PER_CLASS)
    for event_id in ids:
        df = events[event_id].select(cfg.timestamp_col, plot_signal).drop_nulls().collect()
        minutes = (df[cfg.timestamp_col] - df[cfg.timestamp_col][0]).dt.total_seconds() / 60
        ax.plot(minutes, df[plot_signal], linewidth=0.8)
    ax.set_title(f"{cls}: {len(ids)} events")
    ax.set_xlabel("minutes since event start")
np.atleast_1d(axes)[0].set_ylabel(plot_signal)
fig.tight_layout()

## 6. Features: one row per event

Each signal is aggregated over the event window (mean, std, percentiles). The
aggregates see *levels*; to also see *dynamics*, per-sample features are derived
first: the step between consecutive samples (`diff1`) and a rolling standard
deviation. Add your own with the `@feature` decorator or the other `make_*`
helpers in `tessa.features.builtins`.

A signal from a slower source sits on the faster source's time grid with nulls in
between, so its per-sample dynamics features come out empty. The next cell lists
and drops such all-null columns rather than letting them reach the analyses.

In [ ]:
features = FeatureRegistry()  # a fresh registry, so re-running this cell is safe
for s in signals:
    make_first_difference(s, registry=features)
    make_rolling_std(s, window=10, registry=features)  # TODO: window in samples

period = to_period(
    events,
    cfg,
    aggregators=["mean", "std", "p05", "p95"],
    feature_registry=features,
)
empty = [c for c in period.columns if period[c].null_count() == period.height]
period = period.drop(empty)
print(f"{period.height} events x {period.width} columns; dropped all-null: {empty}")
period.head()

## 7. Analyses

`Run` holds the table and the question. Cross-validation folds are grouped by
asset, so a model is always scored on assets it never saw in training: an asset
signature cannot pass for a class signature.

**Separability first.** It answers "can *any* model tell the classes apart
better than chance?" with a permutation test. If the verdict is "not separable",
the feature ranking below it ranks noise.

In [ ]:
run = Run(
    period,
    target_col=cfg.class_col,
    cfg=cfg,
    label_filter={cfg.class_col: CLASSES},
    # Drop features with many nulls instead of dropping the events holding them.
    null_policy=NullPolicy("drop_features"),
    # Label-sheet columns stay in the table for filtering and stratifying, but
    # never become features, even when numeric: they would leak the label.
    label_cols=events.label_cols,
)

# 100 permutations put the smallest reachable p-value near 0.01; raise it for the
# final version. This is the slowest cell (tens of seconds on the example data).
separability = run.separability(
    n_permutations=100,
    rf_params=dict(n_estimators=100, n_jobs=-1, random_state=RANDOM_STATE),
)
separability.frames["summary"]

In [ ]:
# Which features carry the difference: a blend of model-based and univariate rankings
importance = run.importance()
importance.frames["table"].head(15)

In [ ]:
# Out-of-fold classifier: how well, and which classes get confused
cv = run.cv_classifier()
print(cv.summary(max_rows=5))

In [ ]:
# Does the answer hold within each stratum? (Too few events in a stratum -> no verdict.)
if STRATIFY_BY:
    for (value,), part in period.group_by(STRATIFY_BY, maintain_order=True):
        counts = part[cfg.class_col].value_counts().sort(cfg.class_col).rows()
        if min(n for _, n in counts) < 10:
            print(f"{STRATIFY_BY}={value}: too few events {counts}, skipped")
            continue
        verdict = (
            Run(part, target_col=cfg.class_col, cfg=cfg, label_filter={cfg.class_col: CLASSES},
                null_policy=NullPolicy("drop_features"), label_cols=events.label_cols)
            .separability(n_permutations=50, rf_params=dict(n_estimators=100, n_jobs=-1,
                          random_state=RANDOM_STATE))
            .frames["summary"]
        )
        print(f"{STRATIFY_BY}={value}: {counts}")
        print(verdict[["cv_balanced_accuracy", "perm_p_value", "verdict"]].to_string(index=False))

In [ ]:
# The curated figures of every analysis run so far (the same ones the report shows)
from IPython.display import display

for name, figures in run.figures().items():
    for title, fig in figures:
        print(f"{name}: {title}")
        display(fig)
        plt.close(fig)  # already displayed; stops the inline backend showing it twice

## 8. Save, report, dashboard

- `run.save()` writes every result as parquet plus a manifest.
- `run.report()` writes one self-contained HTML file: the thing to send to
  people who will not open a notebook.
- Browse saved runs with `uv run tessa-dashboard outputs/runs`.

In [ ]:
run_dir = run.save(OUTPUT_DIR / "runs", name="tp_vs_fp")
report = run.report(OUTPUT_DIR / "tp_vs_fp_report.html", title="TP vs FP")
print("saved run :", run_dir)
print("report    :", report)